# Atlas Raman — Environment & Dataset Inventory

**Goal.** Reproduce the dataset summary: **87 files, 7,122 QC-passed spectra, 987 wavenumber bins**. This notebook is read-only over `data_cache/` — no data is regenerated.

## How to run

From the repository root (`NomadX/`):

```bash
ln -s /Users/devashishthapliyal/Documents/NomadX/data_cache data_cache
ln -s /Users/devashishthapliyal/Documents/NomadX/.venv .venv
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    FINAL/notebooks/01_environment_inventory.ipynb
```

Unit 1 only needs `data_cache/` — no `Atlas Data/` symlink required.

In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

print(f'Python: {sys.version.split()[0]}')
print(f'Executable: {sys.executable}')

pkgs = ['numpy', 'pandas', 'scipy', 'scikit-learn',
        'xgboost', 'pyMCR', 'PyWavelets', 'pybaselines']
for p in pkgs:
    try:
        print(f'{p:>14s}  {version(p)}')
    except PackageNotFoundError:
        print(f'{p:>14s}  (not installed)')


Python: 3.12.12
Executable: /Users/devashishthapliyal/Documents/NomadX/.venv/bin/python
         numpy  1.26.4
        pandas  2.2.2
         scipy  1.13.1
  scikit-learn  1.5.1
       xgboost  2.1.0
         pyMCR  0.5.1
    PyWavelets  1.9.0
   pybaselines  1.1.0


In [2]:
import pandas as pd

meta = pd.read_parquet('../../data_cache/metadata.parquet')
print(f'metadata.parquet shape: {meta.shape}')
meta.head(5)


metadata.parquet shape: (87, 23)


,file_id,file_path,primary_class,subclass,n_pixels,grid_nx,grid_ny,header_numx,header_numy,xsize,...,acquisition_date,ac_calibration_date,wn_start,wn_end,is_complete_scan,file_sha256,file_mtime,file_size_bytes,fatal_errors,warnings
0,R372_100_10000ms_260306,H20/R372_100_10000ms_260306.xls,H2O,None,200,14,15,14,15,0.5,...,06.03.2026 09:47:57,05.03.2026 07:48:36,75.87,3498.75,True,b805bf469019d2932ba3e1e3e31d540a859e268dfeacbb...,1.772821e+09,3953545,[],"[""pixel_capped:210->200""]"
1,R435_100_10000ms_260313,H20/R435_100_10000ms_260313.xls,H2O,None,81,9,9,9,9,0.5,...,13.03.2026 09:56:45,13.03.2026 08:36:10,76.06,3498.95,True,3454c731bec5530ab5783d555f97f682f10202274cdf57...,1.773422e+09,1534343,[],[]
2,R436_100_10000ms_260313,H20/R436_100_10000ms_260313.xls,H2O,None,81,9,9,9,9,0.5,...,13.03.2026 10:20:34,13.03.2026 08:36:10,76.06,3498.95,True,7eab47d262b7f02f75b450769a05e44444a7f7553c457a...,1.773423e+09,1532516,[],[]
3,R437_100_10000ms_260313,H20/R437_100_10000ms_260313.xls,H2O,None,81,9,9,9,8,0.5,...,13.03.2026 10:45:07,13.03.2026 08:36:10,76.06,3498.95,True,6f3c60ab9f6cbc228ffe9a993b047ec77aa90de1ecc2ad...,1.773425e+09,1532031,[],[]
4,R438_100_10000ms_260313,H20/R438_100_10000ms_260313.xls,H2O,None,81,9,9,9,9,0.5,...,13.03.2026 11:10:55,13.03.2026 08:36:10,76.06,3498.95,True,31af74a80d3042d7d64834fc29d1bd08b4c819f6e024a2...,1.773426e+09,1532077,[],[]


In [3]:
by_class = (meta.groupby('primary_class')['file_id']
                .nunique()
                .rename('n_files')
                .sort_values(ascending=False))
print('Files by primary class:')
print(by_class.to_string())
print(f'\nTotal files: {meta["file_id"].nunique()}')


Files by primary class:
primary_class
STEC          27
Salmonella    27
Non-STEC      25
H2O            8

Total files: 87


In [4]:
subclass_display = meta['subclass'].fillna('H2O (blank)')
by_sub = (meta.assign(subclass_display=subclass_display)
              .groupby('subclass_display')['file_id']
              .nunique()
              .rename('n_files')
              .sort_index())
print('Files by subclass (9 bacterial + 1 blank):')
print(by_sub.to_string())


Files by subclass (9 bacterial + 1 blank):
subclass_display
83972          8
ATCC25922      9
Dublin         9
H2O (blank)    8
Heidelburg     9
K-12           8
O103H2         9
O121H19        9
O157H7         9
Typhimurium    9


In [5]:
import numpy as np

qc_mask = np.load('../../data_cache/qc_mask.npy')
n_total = int(qc_mask.size)
n_pass = int(qc_mask.sum())
print(f'qc_mask shape: {qc_mask.shape}  dtype: {qc_mask.dtype}')
print(f'QC-passed spectra: {n_pass}')
print(f'QC-pass rate: {n_pass / n_total:.3%}  ({n_pass}/{n_total})')


qc_mask shape: (7999,)  dtype: bool
QC-passed spectra: 7122
QC-pass rate: 89.036%  (7122/7999)


In [6]:
wn = np.load('../../data_cache/wavenumber_axis_preprocessed.npy')
print(f'wavenumber bins: {len(wn)}')
print(f'range: {wn.min():.1f} – {wn.max():.1f} cm^-1')
print(f'mean step: {np.diff(wn).mean():.3f} cm^-1')


wavenumber bins: 987
range: 400.4 – 3049.2 cm^-1
mean step: 2.686 cm^-1


## Summary

**Inventory: 87 files, 7,122 QC-passed spectra, 987 preprocessed wavenumber bins (~400–3050 cm⁻¹; fingerprint region ~600–1750 cm⁻¹ used for modeling).**